<a href="https://colab.research.google.com/github/Seripro/c-learning/blob/main/gpu_learning/monte_carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun May 24 01:30:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%%writefile monte_carlo.cu
#include <stdio.h>
#include <stdlib.h>

// ==========================================
// ★GPU用の簡易サイコロ（乱数生成器）
// HPCのシミュレーションでは乱数の質も重要だが、今回は学習用にシンプルなものを使う
// ==========================================
__device__ float generate_rand(unsigned int *seed) {
    *seed = (1664525 * (*seed) + 1013904223);
    return (float)(*seed & 0x00FFFFFF) / (float)0x01000000; // 0.0 〜 1.0 の小数を返す
}

// ==========================================
// ★GPUカーネル：10万人の作業員がダーツを投げる！
// ==========================================
__global__ void monte_carlo_pi(long long throws_per_thread, unsigned long long *d_total_in_circle) {
    // 自分の出席番号（通し番号）を計算
    long long i = blockDim.x * blockIdx.x + threadIdx.x;

    // スレッドごとに違うサイコロの振り方をするための「種（シード）」を作る
    unsigned int seed = i * 1234567;

    // 自分専用のホワイトボード（ローカル変数）を用意
    unsigned long long local_in = 0;

    // 指定された回数だけ、ひたすらダーツを投げる！
    for (long long t = 0; t < throws_per_thread; t++) {
        float x = generate_rand(&seed); // X座標（0.0 ~ 1.0）
        float y = generate_rand(&seed); // Y座標（0.0 ~ 1.0）

        // x^2 + y^2 <= 1 なら、円の中に入ったと判定！
        if (x * x + y * y <= 1.0f) {
            local_in++;
        }
    }

    // ★出た！これがレースコンディションを防ぐ必殺技「Atomic演算」だ！
    // 自分が投げ終わった結果（local_in）を、軍団全体の合計箱に安全に足し込む
    atomicAdd(d_total_in_circle, local_in);
}

int main(void) {
    // 1小隊256人 × 3907小隊 = 約100万人の作業員を出撃させる
    int threadsPerBlock = 256;
    int blocksPerGrid = 3907;
    long long total_threads = threadsPerBlock * blocksPerGrid;

    // 1人あたり1000回ダーツを投げる
    long long throws_per_thread = 1000;

    // 全体で投げるダーツの総数（約10億回！！）
    long long total_throws = total_threads * throws_per_thread;

    printf("約 %lld 個のスレッドが、それぞれ %lld 回ダーツを投げます。\n", total_threads, throws_per_thread);
    printf("合計 %lld 回の計算をGPUで一斉スタート！...\n", total_throws);

    // 本社（CPU）と工場（GPU）のメモリ確保
    unsigned long long h_total_in_circle = 0;
    unsigned long long *d_total_in_circle;

    cudaMalloc((void**)&d_total_in_circle, sizeof(unsigned long long));
    cudaMemcpy(d_total_in_circle, &h_total_in_circle, sizeof(unsigned long long), cudaMemcpyHostToDevice);

    // ★GPU出撃命令！
    monte_carlo_pi<<<blocksPerGrid, threadsPerBlock>>>(throws_per_thread, d_total_in_circle);

    // 結果を工場から本社へ持ち帰る
    cudaMemcpy(&h_total_in_circle, d_total_in_circle, sizeof(unsigned long long), cudaMemcpyDeviceToHost);

    // 円周率の計算
    // π = 4 * (円に入った数 / 投げた総数)
    double pi_estimate = 4.0 * (double)h_total_in_circle / (double)total_throws;

    printf("GPUが弾き出した円周率: %f\n", pi_estimate);

    cudaFree(d_total_in_circle);

    return 0;
}

Writing monte_carlo.cu


In [2]:
!wget -q https://developer.download.nvidia.com/hpc-sdk/25.1/nvhpc_2025_251_Linux_x86_64_cuda_multi.tar.gz
!tar -xzf nvhpc_2025_251_Linux_x86_64_cuda_multi.tar.gz

In [3]:
!nvcc monte_carlo.cu -o monte_carlo

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./monte_carlo

約 1000192 個のスレッドが、それぞれ 1000 回ダーツを投げます。
合計 1000192000 回の計算をGPUで一斉スタート！...
GPUが弾き出した円周率: 3.141603
